In [1]:
# Step 0: Importing the functions

# Tudat imports for propagation and estimation
from tudatpy.interface import spice
from tudatpy import dynamics
from tudatpy.dynamics import environment_setup
from tudatpy.dynamics import propagation_setup, parameters_setup, simulator
from tudatpy import estimation
from tudatpy.estimation import observable_models_setup, observable_models, observations_setup, observations, estimation_analysis
from tudatpy.astro.time_representation import DateTime

# import MPC interface
from tudatpy.data.mpc import BatchMPC

# import SBDB interface
from tudatpy.data.sbdb import SBDBquery

# other useful modules
import numpy as np
import datetime

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.cm as cm

# additional things for the asteroids
from tudatpy import constants
import os           # for the extraction of asteroid kernels
# from astroquery.jplhorizons import Horizons     # for extracting Cartesian coordinates


# Load SPICE Kernel
# SPICE KERNELS
spice.load_standard_kernels()

In [81]:
# Step 1: Set constants

# Setting some constants
# target_mpc_code = 704       # Eros
# The 9 asteroids
target_mpc_code_list = [1566, 66146, 66391, 437844, 137924, 138127, 480883, 468468, 364136]
# target_mpc_code_list = [ 66146]

observations_start = datetime.datetime(2020, 1, 1)
observations_end = datetime.datetime(2025, 1, 1)

# number of iterations for our estimation
number_of_pod_iterations = 6

# timestep of 20 hours for our estimation
timestep_global = 20 * 3600.0

# 1 month time buffer used to avoid interpolation errors:
time_buffer = 1 * 31 * 86400.0

# define the frame origin and orientation.
global_frame_origin = "SSB"
global_frame_orientation = "J2000"

In [82]:
target_name_list = []
target_spkid_list = []

for id in target_mpc_code_list:
    target_sbdb_id = SBDBquery(id)

    target_spkid_element = target_sbdb_id.codes_300_spkid

    obj = target_sbdb_id.query.get("object", {})
    # print(obj.keys())

    target_name_element = (
        obj.get("shortname")
        or obj.get("fullname")
        or obj.get("des")
        or str(id)
    )

    target_name_list.append(target_name_element)
    target_spkid_list.append(target_spkid_element)

    print(f"SPK ID for {target_name_element} is: {target_spkid_element}")

SPK ID for 1566 Icarus is: 2001566
SPK ID for 66146 (1998 TU3) is: 2066146
SPK ID for 66391 Moshup is: 2066391
SPK ID for 437844 (1999 MN) is: 2437844
SPK ID for 137924 (2000 BD19) is: 2137924
SPK ID for 138127 (2000 EE14) is: 2138127
SPK ID for 480883 (2001 YE4) is: 2480883
SPK ID for 468468 (2004 KH17) is: 2468468
SPK ID for 364136 (2006 CJ) is: 2364136


In [83]:
# Step 3: Pulling out data from the Batch / retrieving data

batch = BatchMPC()

batch.get_observations(target_mpc_code_list)
batch.filter(
    epoch_start=observations_start,
    epoch_end=observations_end,
)

batch.summary()

/home/emmabob/miniconda3/envs/tudat-space/lib/python3.10/site-packages/tudatpy/data/mpc/mpc.py:866: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['1566' '1566' '1566' ... '1566' '1566' '1566']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  obs.loc[:, "number"] = obs.number.astype(str)
/home/emmabob/miniconda3/envs/tudat-space/lib/python3.10/site-packages/tudatpy/data/mpc/mpc.py:866: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['66146' '66146' '66146' ... '66146' '66146' '66146']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  obs.loc[:, "number"] = obs.number.astype(str)
/home/emmabob/miniconda3/envs/tudat-space/lib/python3.10/site-packages/tudatpy/data/mpc/mpc.py:866: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a fut


   Batch Summary:
1. Batch includes 9 minor planets:
   ['1566', '66146', '66391', 'h7844', 'D7924', 'D8127', 'm0883', 'k8468', 'a4136']
2. Batch includes 2896 observations, including 509 observations from space telescopes
3. The observations range from 2020-01-01 13:20:29.068798 to 2024-12-31 15:13:58.079984
   In seconds TDB since J2000: 631156898.2527316 to 788930107.263878
   In Julian Days: 2458850.055892 to 2460676.1347
4. The batch contains observations from 57 observatories, including 2 space telescopes



In [84]:
# Include space-telescopes:

print("Summary of space telescopes in batch:")
print(batch.observatories_table(only_space_telescopes=True))

Summary of space telescopes in batch:
     Code  Name  count
1239  C51  WISE   57.0
1245  C57  TESS  452.0


In [85]:
obs_by_WISE = (
    batch.table.query("observatory == 'C51'")
    .loc[:, ["number", "epochUTC", "RA", "DEC"]]
    .iloc[[0, -1]]
)

print("\nInitial and Final Observations by WISE:")
print(obs_by_WISE)


Initial and Final Observations by WISE:
     number                   epochUTC        RA       DEC
1643   1566 2020-09-07 17:28:11.423991  4.899616 -0.959079
418   m0883 2021-12-26 21:22:30.604784  0.268088  0.533647


In [86]:
# Step 4: Set up the environment:

# List the bodies for our environment
bodies_to_create = [
    "Sun",
    "Mercury",
    "Venus",
    "Earth",
    "Moon",
    "Mars",
    "Jupiter",
    "Saturn",
    "Uranus",
    "Neptune",
]

# Create system of bodies
body_settings = environment_setup.get_default_body_settings(
    bodies_to_create, global_frame_origin, global_frame_orientation
)

bodies = environment_setup.create_system_of_bodies(body_settings)

# Retrieve Eros' body name from BatchMPC and set its centre to enable its propagation
bodies_to_propagate = batch.MPC_objects
print(bodies_to_propagate)

# Central bodies needs to be as big as bodies_to_propagate
central_bodies = ['SSB'] * len(batch.MPC_objects)

['1566', '66146', '66391', 'h7844', 'D7924', 'D8127', 'm0883', 'k8468', 'a4136']


In [87]:
# Step 5: Convert the observations to Tudat

# We filter out the space based observations
# This is also where we'd add bias settings
# For this example we use: the plain angular position observation settings

# Transform the MPC observations into a tudat compatible format.
# note that we explicitly exclude all satellite observations in this step by setting included satellites to None.
observation_collection = batch.to_tudat(bodies=bodies, included_satellites=None)

# set create angular_position settings for each link in the list.
observation_settings_list = list()
link_list = list(
    observation_collection.get_link_definitions_for_observables(
        observable_type=observable_models_setup.model_settings.angular_position_type
    )
)

for link in link_list:
    # add optional bias settings here
    observation_settings_list.append(
        observable_models_setup.model_settings.angular_position(link, bias_settings=None)
    )
# Retrieve the first and final observation epochs and add the buffer
epoch_start_nobuffer = batch.epoch_start
epoch_end_nobuffer = batch.epoch_end

epoch_start_buffer = epoch_start_nobuffer - time_buffer
epoch_end_buffer = epoch_end_nobuffer + time_buffer

/home/emmabob/miniconda3/envs/tudat-space/lib/python3.10/site-packages/tudatpy/data/mpc/mpc.py:147: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  bias_dataframe = bias_dataframe.stack(level=0)


In [88]:
# Step 6: Create the acceleration settings

# Define accelerations
accelerations = {
    "Sun": [
        propagation_setup.acceleration.point_mass_gravity(),
        propagation_setup.acceleration.relativistic_correction(use_schwarzschild=True),
    ],
    "Mercury": [propagation_setup.acceleration.point_mass_gravity()],
    "Venus": [propagation_setup.acceleration.point_mass_gravity()],
    "Earth": [propagation_setup.acceleration.point_mass_gravity()],
    "Moon": [propagation_setup.acceleration.point_mass_gravity()],
    "Mars": [propagation_setup.acceleration.point_mass_gravity()],
    "Jupiter": [propagation_setup.acceleration.point_mass_gravity()],
    "Saturn": [propagation_setup.acceleration.point_mass_gravity()],
    "Uranus": [propagation_setup.acceleration.point_mass_gravity()],
    "Neptune": [propagation_setup.acceleration.point_mass_gravity()],
}

# Set up the accelerations settings for each body, in this case only Eros
acceleration_settings = {}
for body in batch.MPC_objects:
    acceleration_settings[str(body)] = accelerations

# create the acceleration models.
acceleration_models = propagation_setup.create_acceleration_models(
    bodies, acceleration_settings, bodies_to_propagate, central_bodies
)

In [89]:
# Step 7.5: Extract the ephemeris for the 9 asteroids
# From my local kenrel

# Extract the ephemeris for the target asteroids for the initial guess

kernel_directory_21_asteroids = "/home/emmabob/Bachelor_Project/asteroid_kernels/9_asteroids_SPK" 

In [90]:
print(target_spkid_list)
target_spk_local_kernel = ['20001566', '20066146', '20066391', '20437844', '20137924', '20138127', '20480883', '20468468', '20364136']

['2001566', '2066146', '2066391', '2437844', '2137924', '2138127', '2480883', '2468468', '2364136']


In [91]:
# Loop through the dictionary and load the corresponding .bsp file
for name, id in zip(target_name_list, target_spk_local_kernel):
    
    # Calculate the NAIF ID from the data list
    # E.g. Juno has 20000003 (ID: 3)
    naif_id = int(id)
    
    # Then extract the ephemeris for each asteroid
    kernel_path = os.path.join(kernel_directory_21_asteroids, f"{naif_id}.bsp")
    if os.path.exists(kernel_path):
        spice.load_kernel(kernel_path)
        print(f"Successfully loaded the kernel for {name} using file: {naif_id}.bsp")
    else:
        print(f"WARNING: Could not find {name}. Its kernel was not found at {kernel_path}. Make sure the file exists.")


Successfully loaded the kernel for 1566 Icarus using file: 20001566.bsp
Successfully loaded the kernel for 66146 (1998 TU3) using file: 20066146.bsp
Successfully loaded the kernel for 66391 Moshup using file: 20066391.bsp
Successfully loaded the kernel for 437844 (1999 MN) using file: 20437844.bsp
Successfully loaded the kernel for 137924 (2000 BD19) using file: 20137924.bsp
Successfully loaded the kernel for 138127 (2000 EE14) using file: 20138127.bsp
Successfully loaded the kernel for 480883 (2001 YE4) using file: 20480883.bsp
Successfully loaded the kernel for 468468 (2004 KH17) using file: 20468468.bsp
Successfully loaded the kernel for 364136 (2006 CJ) using file: 20364136.bsp


In [92]:
# Step 7: Retrieve the initial guess for the asteroids

rng = np.random.default_rng(seed=1)

asteroid_states = []

initial_position_offset = 1e3
initial_velocity_offset = 1

for spk_id, name in zip(target_spk_local_kernel, target_name_list):

    # 1. Get SPICE initial state
    initial_states = spice.get_body_cartesian_state_at_epoch(
        spk_id,
        global_frame_origin,
        global_frame_orientation,
        "NONE",
        epoch_start_buffer,
    )

    # 2. Create noisy initial guess
    initial_guess = initial_states.copy()
    initial_guess[0:3] += (2 * rng.random(3) - 1) * initial_position_offset
    initial_guess[3:6] += (2 * rng.random(3) - 1) * initial_velocity_offset

    # 3. Store everything in a structured way
    asteroid_states.append({
        "spkid": spk_id,
        "name": name,
        "true_state": initial_states,
        "initial_guess": initial_guess,
        "error": initial_guess - initial_states
    })

    print(f"{name} ({spk_id})")
    print("Error:", asteroid_states[-1]["error"])
    print("-" * 50)

1566 Icarus (20001566)
Error: [ 2.36432495e+01  9.00927399e+02 -7.11680771e+02  8.97298894e-01
 -3.76337096e-01 -1.53347102e-01]
--------------------------------------------------
66146 (1998 TU3) (20066146)
Error: [ 6.55405182e+02 -1.81601730e+02  9.91873741e+01 -9.44881774e-01
  5.07026217e-01  7.62866264e-02]
--------------------------------------------------
66391 Moshup (20066391)
Error: [-3.40536568e+02  5.76857407e+02 -3.93610344e+02 -9.30042210e-02
 -7.31916606e-01 -1.93774027e-01]
--------------------------------------------------
437844 (1999 MN) (20437844)
Error: [-5.93089519e+02 -4.75373322e+02  5.00729347e+02 -4.39182484e-01
 -2.96180511e-02  9.61474400e-01]
--------------------------------------------------
137924 (2000 BD19) (20137924)
Error: [ 9.23314392e+02  4.49579895e+02  8.24537048e+01 -4.46217592e-01
 -6.78695982e-01  9.39850826e-01]
--------------------------------------------------
138127 (2000 EE14) (20138127)
Error: [ 3.21371689e+01 -7.68268768e+02  2.46979511e

In [93]:
# Step 7.75: Create one combined initial guess (6*9 = 54)

combined_initial_guess = np.hstack(
    [obj["initial_guess"] for obj in asteroid_states]
)

print(combined_initial_guess.shape)

(54,)


In [94]:
# Step 8: Finalise the propagation setup

# Create numerical integrator settings
integrator_settings = propagation_setup.integrator.runge_kutta_variable_step_size(
    timestep_global,
    propagation_setup.integrator.CoefficientSets.rkf_78,
    timestep_global,
    timestep_global,
    1.0,
    1.0,
)
# Terminate at the time of oldest observation
termination_condition = propagation_setup.propagator.time_termination(epoch_end_buffer)

# To create propagator:

bodies_to_integrate = [
        obj["name"] for obj in asteroid_states      # Grab all 9 asteroids
    ],

propagator_settings = propagation_setup.propagator.translational(
    central_bodies = central_bodies,
    acceleration_models = acceleration_models,
    bodies_to_integrate = bodies_to_propagate,
    initial_states = combined_initial_guess,        # Grab the (54,) initial guess
    initial_time = epoch_start_buffer,
    integrator_settings = integrator_settings,
    termination_settings = termination_condition,
)



In [95]:
# Step 9: Setting Up the Estimation
# The observation collection, the environment and propagations settings are ready
# Commence estimation

# Here: In this example we will simply estimate the position of the 9 asteroids 
# and as such only include an initial states parameter.

In [96]:
# Setup parameters settings to propagate the state transition matrix
parameter_settings = parameters_setup.initial_states(
    propagator_settings, bodies
)

# Create the parameters that will be estimated
parameters_to_estimate = parameters_setup.create_parameter_set(
    parameter_settings, bodies, propagator_settings
)

# Max iterations was set to 6

# Set up the estimator
estimator = estimation_analysis.Estimator(
    bodies=bodies,
    estimated_parameters=parameters_to_estimate,
    observation_settings=observation_settings_list,
    propagator_settings=propagator_settings,
    integrate_on_creation=True,
)

# provide the observation collection as input, and limit number of iterations for estimation.
pod_input = estimation_analysis.EstimationInput(
    observations_and_times=observation_collection,
    convergence_checker=estimation.estimation_analysis.estimation_convergence_checker(
        maximum_iterations=number_of_pod_iterations,
    ),
)

# Set methodological options
pod_input.define_estimation_settings(reintegrate_variational_equations=True)

In [97]:
print(parameters_to_estimate.parameter_set_size)
print(bodies_to_integrate)
print(observation_collection.concatenated_observations.shape)

54
(['1566 Icarus', '66146 (1998 TU3)', '66391 Moshup', '437844 (1999 MN)', '137924 (2000 BD19)', '138127 (2000 EE14)', '480883 (2001 YE4)', '468468 (2004 KH17)', '364136 (2006 CJ)'],)
(4774,)


In [98]:
print("Bodies propagated:")
print(bodies_to_integrate)

print("Number of estimated parameters:")
print(parameters_to_estimate.parameter_set_size)

print("Combined state size:")
print(len(combined_initial_guess))

Bodies propagated:
(['1566 Icarus', '66146 (1998 TU3)', '66391 Moshup', '437844 (1999 MN)', '137924 (2000 BD19)', '138127 (2000 EE14)', '480883 (2001 YE4)', '468468 (2004 KH17)', '364136 (2006 CJ)'],)
Number of estimated parameters:
54
Combined state size:
54


In [99]:
# Step 10: Performing the estimation:

# Perform the estimation
pod_output = estimator.perform_estimation(pod_input)

Calculating residuals and partials 4774
Current residual: 1.6912
Parameter update -2.8526e+11  6.06578e+11  4.07452e+11      61105.6      -147065     -1090.61  3.69175e+06 -4.46032e+06 -1.19841e+06      1.64395   -0.0933232     0.299373      22708.6      12915.8      9607.18    0.0692185     0.716714     0.211656 -5.94478e+07   -1.286e+07 -4.03602e+06     -8.57496     -15.8507     -8.39356 -7.45857e+06  1.01148e+06 -1.85634e+06      1.71812     0.103651     -0.66157        19687      13760.5      8554.69    -0.550356    -0.220846    -0.832808  8.43688e+07   2.6323e+08  1.34089e+08     -67.3766      63.4616      24.6934  5.90217e+07   6.4485e+07 -4.77745e+08      90.0614     -24.8628      60.5137      24756.7        48567     -10583.8    -0.571332     0.600413    -0.600855
Calculating residuals and partials 4774
Current residual: 0.792565
Parameter update-2.13618e+13  5.00209e+13 -7.62498e+12      -235291       801879       395916 -3.72977e+06  4.49219e+06  1.23087e+06    -0.706832    -

RuntimeError: Error computing observation of type AngularPosition with link ends transmitter: (1566); receiver: (Earth, 703) at epoch 738025065.684347.
Original error: Error in light-time solution, computing light time with reference epoch 738025065.684347 (DateTime: 2023-05-22 10:57:45.684346914291382) at receiver.
Original error: Error in ephemeris, requesting state at epoch 546484606.253405.
Original error: Error when setting global state of 1566 from ephemeris.
Original error: Error in ephemeris, requesting state at epoch 546484606.253405.
Original error: Error in tabulated ephemeris.
Original error: Error in interpolator, requesting data point outside of boundaries, requested data at 546484606.253405 but limit values are 628478498.252732 and 791630498.252732

In [ ]:
body_settings.get(asteroid).ephemeris_settings

In [100]:
for spk_id, name in zip(target_spk_local_kernel, target_name_list):

    try:
        state = spice.get_body_cartesian_state_at_epoch(
            spk_id,
            global_frame_origin,
            global_frame_orientation,
            "NONE",
            epoch_start_buffer
        )
        print(name, "OK")

    except Exception as e:
        print("FAILED:", name)
        print(e)

1566 Icarus OK
66146 (1998 TU3) OK
66391 Moshup OK
437844 (1999 MN) OK
137924 (2000 BD19) OK
138127 (2000 EE14) OK
480883 (2001 YE4) OK
468468 (2004 KH17) OK
364136 (2006 CJ) OK
